# TimeSformer Ink Detection — GP-Winner Architecture

TimeSformer (*Bertasius et al. 2021*, Facebook AI Research) with **divided space-time attention**
applied to 3D CT Z-stacks for dense ink segmentation.

**Why TimeSformer works here:**
- CT Z-stack = video (Z layers ↔ time frames): same input structure
- Z-depth attention: each (x,y) asks *"is there consistent ink across all 33 Z layers?"*
- Spatial attention: asks *"does surrounding context match a Greek letterform?"*
- **This is the architecture that generated our pseudo-labels** (GP-winner 2023)

**Divided space-time attention** (cost-efficient):
1. Temporal (Z-depth) attention: O(N · T²) — each spatial patch across 33 Z layers
2. Spatial attention: O(T · N²) — all spatial patches within each Z layer

vs full joint attention: O((TN)²) — intractable

**Architecture:**
```
Input (B, Z=33, H=256, W=256)
  -> PatchEmbed p=16 -> tokens (B, 33*256, 192)
  -> Z-depth pos embed + spatial pos embed
  -> 8x TimeSformerBlock (Z-attn + spatial-attn + FFN)
  -> mean over Z -> reshape (B, 192, 16, 16)
  -> ConvTranspose2d decoder x16 -> (B, 1, 256, 256)
```

**Train patches: 256×256**. **Val patches: 256×256**.
Gradient checkpointing enabled by default to manage VRAM.

In [ ]:
# --- 1. Environment detection + data download ---
import os, ssl, sys, urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

IS_KAGGLE = os.path.exists('/kaggle/input') or os.path.exists('/kaggle/working')
print(f'Environment: {"Kaggle" if IS_KAGGLE else "local"}')

ALL_SEGMENTS = [
    ('20231221180251', 3.5), ('20231031143852', 3.7), ('20231016151002', 4.0),
    ('20231106155351', 4.5), ('20230702185753', 4.6), ('20231210121321', 5.1),
    ('20230929220926', 6.1), ('20231022170901', 6.3), ('20231005123336', 8.6),
    ('20231012184424', 9.8), ('20231007101619', 14.2),
]

if IS_KAGGLE:
    DATA_ROOT = Path('/kaggle/working/data/labelled_segments')
    SEGS      = [s for s, _ in ALL_SEGMENTS[:3]]
    MODEL_DIR = Path('/kaggle/working/models')
    PRED_DIR  = Path('/kaggle/working/predictions')
else:
    DATA_ROOT = Path('data/labelled_segments')
    SEGS      = [s for s, _ in ALL_SEGMENTS]
    MODEL_DIR = Path('models')
    PRED_DIR  = Path('predictions')

for _d in [DATA_ROOT, MODEL_DIR, PRED_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

NUM_LAYERS = 33
BASE_URL   = 'https://data.aws.ash2txt.org/samples/PHercParis4/segments'
VOLUME     = '7.91um-54keV-volume-20230205180739.tifs'
LABEL_FN   = ('PHercParis4-{seg}-7.91um-54keV-volume-20230205180739-'
               '20230702185753-tile64-stride16.tif')

_ctx = ssl.create_default_context()
_ctx.check_hostname = False
_ctx.verify_mode    = ssl.CERT_NONE

def _fetch(url, dest, retries=3):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 1000:
        return True
    for i in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=60, context=_ctx) as r, open(dest, 'wb') as f:
                while chunk := r.read(1 << 20):
                    f.write(chunk)
            return True
        except Exception as e:
            print(f'  retry {i+1}: {e}')
    return False

def download_segment(seg):
    d = DATA_ROOT / seg
    _fetch(f'{BASE_URL}/{seg}/ink-detection/' + LABEL_FN.format(seg=seg),
           d / 'ink_labels.tif')
    with ThreadPoolExecutor(max_workers=6) as pool:
        futs = [pool.submit(_fetch,
                            f'{BASE_URL}/{seg}/surface-volumes/{VOLUME}/{i:02d}.tif',
                            d / 'surface_volume' / f'{i:02d}.tif')
                for i in range(NUM_LAYERS)]
        for _ in as_completed(futs): pass
    ok = (len(list((d / 'surface_volume').glob('*.tif'))) == NUM_LAYERS
          and (d / 'ink_labels.tif').exists())
    print(f'  {seg}: {"OK" if ok else "INCOMPLETE"}')
    return ok

need = [s for s in SEGS
        if not ((DATA_ROOT/s/'ink_labels.tif').exists()
                and len(list((DATA_ROOT/s/'surface_volume').glob('*.tif'))) == NUM_LAYERS)]
if need:
    print(f'Downloading {len(need)} segments...')
    for seg in need:
        download_segment(seg)
else:
    print('All segments present — skipping download.')

In [ ]:
# --- 2. Imports & config ---
import math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import tifffile as tff
import matplotlib.pyplot as plt
from torch.utils.data import IterableDataset, DataLoader
from torch.utils.checkpoint import checkpoint
from typing import Iterator, Sequence

try:
    from scipy.ndimage import map_coordinates, gaussian_filter
    ELASTIC = True
except ImportError:
    ELASTIC = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Device: {device}')

VAL_SEG    = SEGS[0]
TRAIN_SEGS = [s for s in SEGS if s != VAL_SEG]

CFG = dict(
    # Patch sizes — 256 for transformer (larger = more tokens, more memory)
    patch_train  = 256,
    patch_val    = 256,
    patch_infer  = 256,
    stride_infer = 128,
    patches_seg  = 400,
    patches_val  = 80,

    # Training
    batch_size   = 2,
    grad_accum   = 4,       # effective batch = 8
    num_epochs   = 60,
    warmup_eps   = 5,
    lr           = 5e-5,    # lower LR for transformer
    wd           = 5e-5,

    # Loss
    focal_alpha  = 0.75,
    focal_gamma  = 2.0,
    ignore_low   = 0.4,
    ignore_high  = 0.6,
    threshold    = 0.4,
    ink_pos      = 0.6,
    ink_neg      = 0.05,

    # Architecture
    z_layers     = NUM_LAYERS,  # 33
    img_size     = 256,
    patch_size   = 16,          # p=16 -> 16x16=256 spatial patches per Z layer
    embed_dim    = 192,         # C — 3 heads x 64 per head
    depth        = 8,           # number of TimeSformer blocks
    num_heads    = 3,
    mlp_ratio    = 4.0,
    drop         = 0.1,
    use_ckpt     = True,        # gradient checkpointing for VRAM efficiency

    device       = device,
)

print(f'Train segs: {len(TRAIN_SEGS)}  Val seg: {VAL_SEG}')
print(f'patch={CFG["patch_train"]}  tokens={CFG["z_layers"]}x{(CFG["img_size"]//CFG["patch_size"])**2}={CFG["z_layers"]*(CFG["img_size"]//CFG["patch_size"])**2}')

In [ ]:
# --- 3. Data loading utilities ---

def load_volume(seg_dir: Path) -> np.ndarray:
    return np.stack([tff.imread(str(seg_dir / 'surface_volume' / f'{i:02d}.tif'))
                     for i in range(NUM_LAYERS)], axis=0)

def load_label(seg_dir: Path) -> np.ndarray:
    return tff.imread(str(seg_dir / 'ink_labels.tif')).astype(np.float32) / 255.0

def derive_mask(vol: np.ndarray) -> np.ndarray:
    return vol.max(axis=0) > 0


def elastic_deform(img, lbl, mask, alpha=50, sigma=5):
    H, W = lbl.shape
    dx = gaussian_filter(np.random.randn(H, W) * alpha, sigma)
    dy = gaussian_filter(np.random.randn(H, W) * alpha, sigma)
    yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')
    cy = np.clip(yy + dy, 0, H - 1).ravel()
    cx = np.clip(xx + dx, 0, W - 1).ravel()
    img_out = np.stack([map_coordinates(img[z], [cy, cx], order=1,
                                        mode='reflect').reshape(H, W)
                        for z in range(img.shape[0])], axis=0)
    lbl_out  = map_coordinates(lbl,               [cy, cx], order=1, mode='reflect').reshape(H, W)
    mask_out = map_coordinates(mask.astype('f4'), [cy, cx], order=0, mode='reflect').reshape(H, W).astype(bool)
    return img_out, lbl_out, mask_out

In [ ]:
# --- 4. Dataset (same augmentations as UNet V2) ---

class SegmentDataset(IterableDataset):
    def __init__(self, data_root, segments, patch_size, patches_per_seg,
                 ink_pos, ink_neg, augment=True, shuffle=True):
        super().__init__()
        self.root    = Path(data_root)
        self.segs    = list(segments)
        self.ps      = patch_size
        self.n       = patches_per_seg
        self.ink_pos = ink_pos
        self.augment = augment
        self.shuffle = shuffle

    def _sample(self, vol, lbl, mask):
        H, W, ps = mask.shape[0], mask.shape[1], self.ps
        for _ in range(50):
            y = random.randint(0, H - ps)
            x = random.randint(0, W - ps)
            m = mask[y:y+ps, x:x+ps]
            if m.mean() < 0.6:
                continue
            img = vol[:, y:y+ps, x:x+ps].astype(np.float32) / 255.0
            lab = lbl[y:y+ps, x:x+ps]
            if random.random() < 0.5 and lab.mean() < self.ink_pos * 0.25:
                continue
            return img, lab, m.astype(np.float32)
        y = random.randint(0, H - ps)
        x = random.randint(0, W - ps)
        return (vol[:, y:y+ps, x:x+ps].astype(np.float32) / 255.0,
                lbl[y:y+ps, x:x+ps],
                mask[y:y+ps, x:x+ps].astype(np.float32))

    def _aug(self, img, lbl, mask):
        if random.random() < 0.5:
            img, lbl, mask = img[:, :, ::-1].copy(), lbl[:, ::-1].copy(), mask[:, ::-1].copy()
        if random.random() < 0.5:
            img, lbl, mask = img[:, ::-1, :].copy(), lbl[::-1].copy(), mask[::-1].copy()
        k = random.randint(0, 3)
        if k:
            img  = np.rot90(img,  k, axes=(1, 2)).copy()
            lbl  = np.rot90(lbl,  k).copy()
            mask = np.rot90(mask, k).copy()
        if random.random() < 0.5:
            img = np.clip(img + np.random.randn(*img.shape).astype(np.float32) * 0.03, 0, 1)
        if random.random() < 0.5:
            img = np.clip(img * random.uniform(0.85, 1.15), 0, 1)
        if random.random() < 0.4:
            idx = random.sample(range(img.shape[0]), random.randint(1, 3))
            img[idx] = 0.0
        if random.random() < 0.3:
            s = random.randint(-2, 2)
            if s:
                img = np.roll(img, s, axis=0)
                if s > 0: img[:s] = 0.0
                else:     img[s:] = 0.0
        if ELASTIC and random.random() < 0.3:
            img, lbl, mask = elastic_deform(img, lbl, mask.astype(bool))
            mask = mask.astype(np.float32)
        return img, lbl, mask

    def __iter__(self) -> Iterator:
        segs = list(self.segs)
        if self.shuffle:
            random.shuffle(segs)
        for sid in segs:
            vol  = load_volume(self.root / sid)
            lbl  = load_label(self.root / sid)
            mask = derive_mask(vol)
            for _ in range(self.n):
                img, y, m = self._sample(vol, lbl, mask)
                if self.augment:
                    img, y, m = self._aug(img, y, m)
                yield (torch.from_numpy(img.copy()),
                       torch.from_numpy(y.copy()),
                       torch.from_numpy(m.copy()))
            del vol, lbl, mask

In [ ]:
# --- 5. TimeSformer architecture ---
# Divided space-time attention:
#   Block 1: Z-depth attention  — each (x,y) attends across all 33 Z layers
#   Block 2: Spatial attention  — all spatial patches attend within each Z layer
# This exactly mirrors how the GP-winner model (Nader/Farritor/Schilliger 2023)
# reasoned about CT Z-stacks: Z depth treated as video frames.


class TimeSformerBlock(nn.Module):
    """
    One TimeSformer block: Z-depth attention, spatial attention, FFN.
    Input / output shape: (B, T*N, C)  where T=Z layers, N=spatial patches.
    """
    def __init__(self, dim, num_heads, mlp_ratio=4.0, drop=0.0):
        super().__init__()
        # Z-depth attention
        self.norm_t  = nn.LayerNorm(dim)
        self.attn_t  = nn.MultiheadAttention(dim, num_heads, dropout=drop, batch_first=True)
        # Spatial attention
        self.norm_s  = nn.LayerNorm(dim)
        self.attn_s  = nn.MultiheadAttention(dim, num_heads, dropout=drop, batch_first=True)
        # Feed-forward
        self.norm_ff = nn.LayerNorm(dim)
        mlp_dim = int(dim * mlp_ratio)
        self.ff = nn.Sequential(
            nn.Linear(dim, mlp_dim), nn.GELU(), nn.Dropout(drop),
            nn.Linear(mlp_dim, dim), nn.Dropout(drop),
        )

    def forward(self, x, T, N):
        # x: (B, T*N, C)
        B, _, C = x.shape

        # --- Z-depth attention ---
        # Reshape so each spatial position is a sequence of T=33 Z tokens
        xt = x.reshape(B, T, N, C).permute(0, 2, 1, 3).reshape(B * N, T, C)
        xt_n = self.norm_t(xt)
        delta_t, _ = self.attn_t(xt_n, xt_n, xt_n)
        xt = xt + delta_t
        x = xt.reshape(B, N, T, C).permute(0, 2, 1, 3).reshape(B, T * N, C)

        # --- Spatial attention ---
        # Reshape so each Z layer is a sequence of N spatial tokens
        xs = x.reshape(B, T, N, C).reshape(B * T, N, C)
        xs_n = self.norm_s(xs)
        delta_s, _ = self.attn_s(xs_n, xs_n, xs_n)
        xs = xs + delta_s
        x = xs.reshape(B, T, N, C).reshape(B, T * N, C)

        # --- FFN ---
        x = x + self.ff(self.norm_ff(x))
        return x


class TimeSformerInkNet(nn.Module):
    """
    TimeSformer for dense ink segmentation on 3D CT Z-stacks.

    Follows the GP-winner architecture class (Nader/Farritor/Schilliger 2023):
    divided space-time attention treats Z-depth as the temporal dimension.

    Input:  (B, Z=33, H, W)
    Output: (B, 1,   H, W) — ink probability logits
    """
    def __init__(self, z_layers=33, img_size=256, patch_size=16,
                 embed_dim=192, depth=8, num_heads=3,
                 mlp_ratio=4.0, drop=0.0, use_checkpoint=True):
        super().__init__()
        assert img_size % patch_size == 0
        self.T           = z_layers
        self.p           = patch_size
        self.use_ckpt    = use_checkpoint
        n_side           = img_size // patch_size  # 16 for 256px/p=16
        self.n_side      = n_side
        N                = n_side * n_side          # 256 spatial patches

        # Patch embedding: one Conv2d applied to each Z slice independently
        self.patch_embed = nn.Conv2d(1, embed_dim, patch_size, patch_size)

        # Learnable positional embeddings
        self.spatial_pos = nn.Parameter(torch.zeros(1, N, embed_dim))
        self.z_pos       = nn.Parameter(torch.zeros(1, z_layers, 1, embed_dim))

        # Transformer blocks
        self.blocks = nn.ModuleList([
            TimeSformerBlock(embed_dim, num_heads, mlp_ratio, drop)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

        # Decoder: log2(patch_size) stages of x2 upsampling
        n_stages = int(math.log2(patch_size))  # 4 for p=16
        ch, layers = embed_dim, []
        for _ in range(n_stages):
            out_ch = max(ch // 2, 16)
            layers += [nn.ConvTranspose2d(ch, out_ch, 2, stride=2),
                       nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
            ch = out_ch
        layers.append(nn.Conv2d(ch, 1, 1))
        self.decoder = nn.Sequential(*layers)

        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.spatial_pos, std=0.02)
        nn.init.trunc_normal_(self.z_pos,       std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')

    def _interp_spatial_pos(self, N_h, N_w):
        N = N_h * N_w
        if N_h == self.n_side and N_w == self.n_side:
            return self.spatial_pos   # (1, N, C)
        C = self.spatial_pos.shape[-1]
        pos = self.spatial_pos.reshape(1, self.n_side, self.n_side, C).permute(0, 3, 1, 2)
        pos = F.interpolate(pos, size=(N_h, N_w), mode='bilinear', align_corners=False)
        return pos.permute(0, 2, 3, 1).reshape(1, N, C)

    def forward(self, x):
        # x: (B, T, H, W)
        B, T, H, W = x.shape
        p = self.p
        N_h, N_w = H // p, W // p
        N = N_h * N_w

        # Embed all Z slices in one batched pass
        tokens = self.patch_embed(x.reshape(B * T, 1, H, W))   # (B*T, C, N_h, N_w)
        C      = tokens.shape[1]
        tokens = tokens.flatten(2).transpose(1, 2)              # (B*T, N, C)
        tokens = tokens.reshape(B, T, N, C)                     # (B, T, N, C)

        # Positional embeddings
        sp = self._interp_spatial_pos(N_h, N_w)                 # (1, N, C)
        tokens = tokens + sp.unsqueeze(1)                       # broadcast over T
        tokens = tokens + self.z_pos[:, :T]                     # broadcast over N
        tokens = tokens.reshape(B, T * N, C)                    # (B, T*N, C)

        # Transformer blocks (gradient checkpointing saves VRAM)
        for block in self.blocks:
            if self.use_ckpt and self.training:
                tokens = checkpoint(block, tokens, T, N, use_reentrant=False)
            else:
                tokens = block(tokens, T, N)

        tokens = self.norm(tokens)                              # (B, T*N, C)

        # Average over Z, reshape to spatial grid, decode
        tokens = tokens.reshape(B, T, N, C).mean(dim=1)        # (B, N, C)
        tokens = tokens.transpose(1, 2).reshape(B, C, N_h, N_w)  # (B, C, N_h, N_w)
        return self.decoder(tokens)                             # (B, 1, H, W)

In [ ]:
# --- 6. Loss + metric ---

def focal_loss(logits, target, alpha=0.75, gamma=2.0):
    bce = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    p   = torch.sigmoid(logits)
    p_t = p * target + (1 - p) * (1 - target)
    a_t = alpha * target + (1 - alpha) * (1 - target)
    return (a_t * (1 - p_t) ** gamma * bce).mean()


def loss_fn(logits, target, mask, cfg):
    logits = logits.squeeze(1)
    sup = mask.bool() & ((target < cfg['ignore_low']) | (target > cfg['ignore_high']))
    if sup.sum() == 0:
        return logits.sum() * 0.0
    fl = focal_loss(logits[sup], target[sup], cfg['focal_alpha'], cfg['focal_gamma'])
    p  = torch.sigmoid(logits)
    m  = sup.float()
    inter = (p * target * m).sum()
    denom = (p * m).sum() + (target * m).sum() + 1e-6
    dice  = 1.0 - (2.0 * inter + 1e-6) / denom
    return 0.5 * fl + 0.5 * dice


@torch.no_grad()
def fbeta(logits, target, mask, beta=0.5, thresh=0.4):
    p  = (torch.sigmoid(logits).squeeze(1) > thresh) & mask.bool()
    t  = (target > 0.5) & mask.bool()
    tp = (p & t).sum().float()
    fp = (p & ~t).sum().float()
    fn = (~p & t).sum().float()
    pr = tp / (tp + fp + 1e-6)
    rc = tp / (tp + fn + 1e-6)
    return ((1 + beta**2) * pr * rc / (beta**2 * pr + rc + 1e-6)).item()


def warmup_cosine(optimizer, warmup_epochs, total_epochs):
    def _lr(ep):
        if ep < warmup_epochs:
            return (ep + 1) / warmup_epochs
        p = (ep - warmup_epochs) / max(total_epochs - warmup_epochs, 1)
        return 0.5 * (1.0 + math.cos(math.pi * p))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, _lr)

In [ ]:
# --- 7. Datasets + DataLoaders ---

train_ds = SegmentDataset(
    DATA_ROOT, TRAIN_SEGS,
    patch_size=CFG['patch_train'], patches_per_seg=CFG['patches_seg'],
    ink_pos=CFG['ink_pos'], ink_neg=CFG['ink_neg'],
    augment=True, shuffle=True,
)
val_ds = SegmentDataset(
    DATA_ROOT, [VAL_SEG],
    patch_size=CFG['patch_val'], patches_per_seg=CFG['patches_val'],
    ink_pos=CFG['ink_pos'], ink_neg=CFG['ink_neg'],
    augment=False, shuffle=False,
)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], num_workers=0,
                          pin_memory=(device == 'cuda'))
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], num_workers=0,
                          pin_memory=(device == 'cuda'))

print(f'Train: {len(TRAIN_SEGS)} segs x {CFG["patches_seg"]} @ {CFG["patch_train"]}px')
print(f'Val:   1 seg x {CFG["patches_val"]} @ {CFG["patch_val"]}px')

In [ ]:
# --- 8. Model + optimizer ---

model = TimeSformerInkNet(
    z_layers     = CFG['z_layers'],
    img_size     = CFG['img_size'],
    patch_size   = CFG['patch_size'],
    embed_dim    = CFG['embed_dim'],
    depth        = CFG['depth'],
    num_heads    = CFG['num_heads'],
    mlp_ratio    = CFG['mlp_ratio'],
    drop         = CFG['drop'],
    use_checkpoint = CFG['use_ckpt'],
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
scheduler = warmup_cosine(optimizer, CFG['warmup_eps'], CFG['num_epochs'])
scaler    = torch.amp.GradScaler(enabled=(device == 'cuda'))

n_params = sum(p.numel() for p in model.parameters())
print(f'TimeSformerInkNet: {n_params/1e6:.2f}M params  device={device}')
print(f'Token count per sample: {CFG["z_layers"]}x{(CFG["img_size"]//CFG["patch_size"])**2} = {CFG["z_layers"]*(CFG["img_size"]//CFG["patch_size"])**2}')

In [ ]:
# --- 9. Training loop ---

hist = {'train_loss': [], 'val_loss': [], 'val_f05': [], 'lr': []}
best_f05 = -1.0

for epoch in range(CFG['num_epochs']):
    model.train()
    total, n = 0.0, 0
    optimizer.zero_grad()

    for i, (img, y, m) in enumerate(train_loader):
        img = img.to(device, non_blocking=True)
        y   = y.to(device, non_blocking=True)
        m   = m.to(device, non_blocking=True)
        with torch.amp.autocast(device_type='cuda', enabled=(device == 'cuda')):
            logits = model(img)
            loss   = loss_fn(logits, y, m, CFG) / CFG['grad_accum']
        scaler.scale(loss).backward()
        if (i + 1) % CFG['grad_accum'] == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        total += loss.item() * CFG['grad_accum']
        n     += 1

    scheduler.step()
    tr_loss = total / max(n, 1)
    cur_lr  = optimizer.param_groups[0]['lr']

    model.eval()
    vl = vf = vn = 0.0
    with torch.no_grad():
        for img, y, m in val_loader:
            img, y, m = img.to(device), y.to(device), m.to(device)
            with torch.amp.autocast(device_type='cuda', enabled=(device == 'cuda')):
                logits = model(img)
                vl    += loss_fn(logits, y, m, CFG).item()
            vf += fbeta(logits, y, m, thresh=CFG['threshold'])
            vn += 1
    va_loss = vl / max(vn, 1)
    va_f05  = vf / max(vn, 1)

    hist['train_loss'].append(tr_loss)
    hist['val_loss'].append(va_loss)
    hist['val_f05'].append(va_f05)
    hist['lr'].append(cur_lr)

    print(f'[{epoch+1:3d}/{CFG["num_epochs"]}] '
          f'train={tr_loss:.4f}  val={va_loss:.4f}  F0.5={va_f05:.4f}  lr={cur_lr:.2e}')

    if va_f05 > best_f05:
        best_f05 = va_f05
        torch.save(model.state_dict(), MODEL_DIR / 'best_timesformer.pth')
        print(f'  saved best F0.5={best_f05:.4f}')

print(f'\nDone. Best val F0.5 = {best_f05:.4f}')

In [ ]:
# --- 10. Training curves ---

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(hist['train_loss'], label='train')
axes[0].plot(hist['val_loss'],   label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend()
axes[1].plot(hist['val_f05'])
axes[1].set_title('Val F0.5'); axes[1].set_xlabel('epoch')
axes[2].plot(hist['lr'])
axes[2].set_title('Learning Rate'); axes[2].set_xlabel('epoch')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'timesformer_curves.png', dpi=100)
plt.show()

In [ ]:
# --- 11. Sliding-window inference ---

@torch.no_grad()
def predict_segment(model, seg_dir, patch_size=256, stride=128, dev='cuda'):
    model.eval()
    vol  = load_volume(Path(seg_dir))
    mask = derive_mask(vol)
    H, W = mask.shape
    prob = np.zeros((H, W), dtype=np.float32)
    wsum = np.zeros((H, W), dtype=np.float32)
    vt   = torch.from_numpy(vol.astype(np.float32) / 255.0)
    for y in range(0, H - patch_size + 1, stride):
        for x in range(0, W - patch_size + 1, stride):
            if not mask[y:y+patch_size, x:x+patch_size].any():
                continue
            patch = vt[:, y:y+patch_size, x:x+patch_size].unsqueeze(0).to(dev)
            with torch.amp.autocast(device_type='cuda', enabled=(dev == 'cuda')):
                out = torch.sigmoid(model(patch)).squeeze().float().cpu().numpy()
            prob[y:y+patch_size, x:x+patch_size] += out
            wsum[y:y+patch_size, x:x+patch_size] += 1.0
    prob = np.where(wsum > 0, prob / np.maximum(wsum, 1e-6), 0.0) * mask
    del vol, vt
    return prob

model.load_state_dict(torch.load(MODEL_DIR / 'best_timesformer.pth', map_location=device))
prob = predict_segment(model, DATA_ROOT / VAL_SEG,
                       patch_size=CFG['patch_infer'],
                       stride=CFG['stride_infer'], dev=device)
np.save(PRED_DIR / f'{VAL_SEG}_timesformer_prob.npy', prob)
tff.imwrite(PRED_DIR / f'{VAL_SEG}_timesformer_prob.tif', (prob * 255).astype(np.uint8))
print(f'Saved {VAL_SEG}  shape={prob.shape}  mean={prob.mean():.4f}')

In [ ]:
# --- 12. Visualization ---

seg_dir = DATA_ROOT / VAL_SEG
gt      = load_label(seg_dir)
ct      = tff.imread(str(seg_dir / 'surface_volume' / '16.tif')).astype(np.float32) / 255.0
mask    = derive_mask(load_volume(seg_dir))
binary  = (prob > CFG['threshold']).astype(np.uint8) * mask

ys, xs  = np.where(mask)
y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()

fig, ax = plt.subplots(2, 3, figsize=(18, 10))
ax[0,0].imshow(ct[y0:y1, x0:x1],   cmap='gray');     ax[0,0].set_title('CT z=16')
ax[0,1].imshow(prob[y0:y1, x0:x1], cmap='viridis');  ax[0,1].set_title('Prediction probability')
ax[0,2].imshow(gt[y0:y1, x0:x1],   cmap='gray');     ax[0,2].set_title('Pseudo-label')
ax[1,0].imshow(binary[y0:y1, x0:x1], cmap='gray');   ax[1,0].set_title(f'Binary t={CFG["threshold"]}')
ov = np.stack([ct[y0:y1, x0:x1]] * 3, axis=-1)
ov[..., 0] = np.maximum(ov[..., 0], binary[y0:y1, x0:x1].astype(np.float32))
ax[1,1].imshow(ov);                                   ax[1,1].set_title('Pred overlay (red)')
ax[1,2].imshow(np.abs(prob[y0:y1, x0:x1] - gt[y0:y1, x0:x1]), cmap='hot')
ax[1,2].set_title('|pred - label|')
for a in ax.ravel(): a.axis('off')
plt.tight_layout()
plt.savefig(PRED_DIR / f'{VAL_SEG}_timesformer_vis.png', dpi=100)
plt.show()